In [ ]:
import random, heapq, math, statistics
import matplotlib.pyplot as plt


# SYSTEM DATA
lam = 42                      # arrivals per hour
mu  = 13.33                   # average service rate/hour/server
T15 = 15/60                   # SLA limit (15 minutes)
cost_per_min = 0.80           # delay cost €/minute/product

# USER INPUT
c = int(input("Enter the number of employees up to 6 (c): "))

# M/M/c ANALYTICS
def mmc_metrics(lam, mu, c):
    Ro = lam / (c * mu)   # Ro = rho

    if Ro >= 1:
        return {"stable": False, "Ro": Ro}

    r = lam / mu
    sum_terms = sum(r**n / math.factorial(n) for n in range(c)) # Sum = Sum (r^n / n!) for n = 0 to c-1
    last = r**c / (math.factorial(c) * (1 - Ro))
    P0 = 1 / (sum_terms + last) # P0 = probability that there are no products in the system

    Pwait = last * P0 # probability that a product waits in the queue
    Lq = P0 * (r**c) * Ro / (math.factorial(c) * (1 - Ro)**2) # Lq = expected number of products in queue
    Wq = Lq / lam # average waiting time (Little's Law)
    W  = Wq + 1/mu # total time in the system

    return {
        "stable": True,
        "Ro": Ro,
        "Pwait": Pwait,
        "Lq": Lq,
        "Wq_h": Wq,
        "W_h": W
    }

# SLA: P(W < 15')
def sla_prob_mmc(lam, mu, c, t_hours):
    m = mmc_metrics(lam, mu, c)
    if not m.get("stable", False):
        return None

    Pwait = m["Pwait"]
    Pnowait = 1 - Pwait

    cdf_service = 1 - math.exp(-mu * t_hours) # When there is no waiting, W = S
                                              # Therefore: P(W < t | no waiting) = P(S < t)
    r1 = c*mu - lam
    r2 = mu
    cdf_sum = 1 - (r2*math.exp(-r1*t_hours) - r1*math.exp(-r2*t_hours)) / (r2 - r1)

    return Pnowait * cdf_service + Pwait * cdf_sum # total SLA probability

# DELAY COST
def delay_cost_per_hour(lam, Wq_hours):
    total_wait_min = lam * Wq_hours * 60 # total waiting minutes per hour
    return total_wait_min * cost_per_min # delay cost / hour

# SIMULATION (M/M/c)
def simulate_mm_c(lam, mu, c, horizon_h=60, warmup_h=10, seed=0):
    random.seed(seed)
    server_free = [0.0]*c # time when each employee/server becomes free
    heapq.heapify(server_free)

    t = 0.0
    waiting_times, system_times = [], []

    while True:
        t += random.expovariate(lam)
        if t > horizon_h:
            break

        free_time = heapq.heappop(server_free)
        start = max(t, free_time)
        w = start - t
        end = start + random.expovariate(mu)
        heapq.heappush(server_free, end)

        if t >= warmup_h:
            waiting_times.append(w)
            system_times.append(end - t)

    return {
        "mean_Wq_min": statistics.mean(waiting_times)*60,
        "mean_W_min": statistics.mean(system_times)*60,
        "p_wait": sum(1 for w in waiting_times if w > 0)/len(waiting_times),
        "p_W_lt_15": sum(1 for x in system_times if x < 15/60)/len(system_times)
    }

def simulate_mm_c_collect(lam, mu, c, horizon_h=60, warmup_h=10, seed=0):
    random.seed(seed)
    server_free = [0.0]*c
    heapq.heapify(server_free)

    t = 0.0
    arrival_times = []
    waiting_times = []
    system_times = []

    while True:
        t += random.expovariate(lam)
        if t > horizon_h:
            break

        free_time = heapq.heappop(server_free)
        start = max(t, free_time)
        w = start - t
        end = start + random.expovariate(mu)
        heapq.heappush(server_free, end)

        if t >= warmup_h:
            arrival_times.append(t)
            waiting_times.append(w)
            system_times.append(end - t)

    return arrival_times, waiting_times, system_times

def run_for_c_values(c_values):
    rows = []
    for c_ in c_values:
        m = mmc_metrics(lam, mu, c_)
        if not m["stable"]:
            rows.append((c_, m["Ro"], None, None, None, None, None))
            continue

        sla = sla_prob_mmc(lam, mu, c_, T15)
        cost = delay_cost_per_hour(lam, m["Wq_h"])
        rows.append((
            c_,
            m["Ro"],
            m["Pwait"],
            m["Wq_h"]*60,
            m["W_h"]*60,
            sla,
            cost
        ))
    return rows

print("\n=== THEORETICAL RESULTS (M/M/{}) ===".format(c))
m = mmc_metrics(lam, mu, c)

if not m["stable"]:
    print("Ro =", round(m["Ro"], 3))
    print("❌ The system is UNSTABLE (Ro ≥ 1)")
else:
    print("Ro =", round(m["Ro"], 3))
    print("P(wait) =", round(m["Pwait"], 4))
    print("Lq =", round(m["Lq"], 3))
    print("Wq =", round(m["Wq_h"]*60, 2), "minutes")
    print("W  =", round(m["W_h"]*60, 2), "minutes")

    sla = sla_prob_mmc(lam, mu, c, T15)
    print("SLA: P(W < 15') =", round(sla, 4))

    cost = delay_cost_per_hour(lam, m["Wq_h"])
    print("Delay cost €/hour =", round(cost, 2))

# ================== RUN SIMULATION ==================
print("\n=== SIMULATION (M/M/{}) ===".format(c))
sim = simulate_mm_c(lam, mu, c)
for k, v in sim.items():
    print(k, "=", round(v, 4))
# ================== COMPARISON OF MULTIPLE c VALUES ==================

c_list = [3,4,5,6]
rows = run_for_c_values(c_list)

print("\nM/M/c comparison")
print("c | Ro | P(wait) | Wq(min) | W(min) | SLA P(W<15') | Cost €/h")
for r in rows:
    c_, Ro, Pwait, Wqmin, Wmin, sla, cost = r
    print(f"{c_:>1} | {Ro:>5.3f} | {('---' if Pwait is None else f'{Pwait:>6.3f}'):>6} | "
          f"{('---' if Wqmin is None else f'{Wqmin:>7.2f}'):>7} | "
          f"{('---' if Wmin is None else f'{Wmin:>6.2f}'):>6} | "
          f"{('---' if sla is None else f'{sla:>11.3f}'):>11} | "
          f"{('---' if cost is None else f'{cost:>8.2f}'):>8}")

# ================== G/G/1 APPROXIMATION ==================
# Approximation formula from the course notes (c = 1)

Ca = 1.1
Cs = 1.5

def gg1_notes_approx(lam, mu, Ca, Cs):
    Ro = lam / mu   # For G/G/1: Ro = lambda / mu

    if Ro >= 1:
        return {
            "stable": False,
            "Ro": Ro,
            "Wq_h": None,
            "W_h": None
        }

    # From the course notes:
    # p^d = Ro^{-1+sqrt(2(c+1))}, for c=1 => p^d = Ro
    pd = Ro

    V = (Ca**2 + Cs**2) / 2  # (Ca^2 + Cs^2)/2

    # W ≈ (1/mu) * V * (pd/(1-Ro)) + 1/mu
    W_h  = (1/mu) * V * (pd/(1 - Ro)) + (1/mu)
    Wq_h = W_h - (1/mu)

    return {
        "stable": True,
        "Ro": Ro,
        "Wq_h": Wq_h,
        "W_h": W_h
    }

gg1 = gg1_notes_approx(lam, mu, Ca, Cs)

print("\n=== G/G/1 APPROXIMATION (from course notes) ===")
print("Ro =", round(gg1["Ro"], 3))

if not gg1["stable"]:
    print("❌ UNSTABLE: Ro ≥ 1")
    print("➡️ The queue grows without bound, so Wq and W are not defined.")
else:
    print("Wq =", round(gg1["Wq_h"]*60, 2), "minutes")
    print("W  =", round(gg1["W_h"]*60, 2), "minutes")
